# Advanced Session: Electrolyte Conductivity Challenge
AI for Materials Science — Hands-on session 5 (optional)

A one-week, Kaggle-style challenge: predict the **ionic conductivity k (mS/cm)** of liquid electrolytes
from their salt, concentration, solvent mixture and temperature. You get the training table with labels, train
a scikit-learn model in this notebook, and upload the trained model as one file, `model.joblib`.

**Site:** https://hos5.tail1aab16.ts.net

- Task: predict ionic conductivity k (mS/cm) of liquid electrolytes on the hidden test rows. Metric: RMSLE (lower is better).
- Open: 2026-11-02 17:30 KST. Deadline: 2026-11-08 23:59 KST. Private results: 2026-11-09 09:00 KST.
- Submission: upload model.joblib, your trained scikit-learn model or pipeline (XGBoost and LightGBM through their scikit-learn classes) saved with joblib.dump. The server loads it and calls predict(test), which must return one predicted k for each test row, in the order of the test rows; test has the feature columns of train, without k.
- Versions: save the model with scikit-learn==1.9.1, xgboost==3.4.1, lightgbm==4.7.0; a model saved with another scikit-learn version is rejected. File size: at most 200 MB. Loading and predicting must finish within 5 minutes, without internet access.
- Limit: 5 scored submissions per student per KST day; runs that fail do not count, up to 10 failed runs a day. The upload time decides whether a submission made the deadline.
- Identity: your student ID and the 6-digit access code sent to you on KLMS. Only your nickname is shown.
- Final ranking uses the private RMSLE of each student's best public submission.
- Too many failed logins trigger a 15-minute cool-down; 30 failures in a day lock that ID for the day.
- Locked? Contact the TA; a server restart clears locks.

---

Run the cells one at a time from the top. Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not defined" error. Lines starting with `##` inside the code are comments written for you; Python does not run them.

## 0. Setup

### 0-1. Install the server's library versions
The server loads your model with scikit-learn 1.9.1, XGBoost 3.4.1 and LightGBM 4.7.0. A saved model
opens reliably only with the versions it was saved with, and the server rejects a model saved with another
scikit-learn version, so the cell below installs exactly these versions before anything else is imported. Run
it first. If it reports other versions, choose Runtime ▸ Restart session and run it again. No GPU is needed.

In [ ]:
!pip install -q scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0   ## the versions the server loads models with
import sklearn, xgboost, lightgbm
print("scikit-learn", sklearn.__version__, " xgboost", xgboost.__version__, " lightgbm", lightgbm.__version__)
assert (sklearn.__version__, xgboost.__version__, lightgbm.__version__) == ("1.9.1", "3.4.1", "4.7.0"), \
    "other versions are loaded: choose Runtime > Restart session, then run this cell again"

### 0-2. Download the course material

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries

In [ ]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

np.random.seed(0)

### 0-4. Set the file paths

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
print("Data folder exists:", DATA_DIR.exists())

## A. The data

### A-1. Load the training table
`train` has the target column `k` and the feature columns. The test rows stay on the server: your
model's `predict` receives them as a table with the same feature columns and no `k`.

In [ ]:
train = pd.read_csv(DATA_DIR / "train_set.csv")
print("train:", train.shape)
train.head()

### A-2. What the columns mean
- `T` — temperature in kelvin.
- `c` — lithium-salt concentration, in the unit given by `c units` (mol/kg or mol/L).
- `salt` — the lithium salt (LiPF6, LiBF4, LiBOB, ...).
- `solvent ratio type` — whether the solvent columns are weight (`w`), volume (`v`) or mole (`mol`) fractions.
- 38 solvent columns (EC, PC, DMC, EMC, DEC, DME, ...) — the fraction of each solvent; each row sums to 1.
- `k` — ionic conductivity in mS/cm (train only). Most values are below 15 mS/cm.

The test rows are new formulations: no salt/solvent/concentration combination in the test set appears in
`train`, so you predict new mixtures, not new temperatures of a mixture you already know.

During the week the leaderboard shows your score on 30% of the test formulations (public). The final ranking
uses the other 70% (private), revealed after the deadline.

### A-3. The target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(train["k"], bins=60)
ax.set_xlabel("ionic conductivity k (mS/cm)"); ax.set_ylabel("rows")
ax.set_title("Training set")
fig.savefig(OUTPUT / "A3_k_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print(train["k"].describe().round(3))

## B. Features
Three columns are text (categories) and the rest are numbers. Most models need numbers only, so how you turn the
text columns into numbers is part of your model: fit that step on the training rows only, and make sure a
category that appears only in the test rows does not crash it.

In [ ]:
CATEGORICAL = ["salt", "c units", "solvent ratio type"]
NUMERIC = [c for c in train.columns if c not in CATEGORICAL + ["k"]]
X, y = train.drop(columns="k"), train["k"]
print(len(NUMERIC), "numeric columns +", len(CATEGORICAL), "categorical columns")

## C. Validate the way the leaderboard scores

### C-1. Formulations as groups
The leaderboard's hidden split keeps every row of one formulation (salt, concentration, unit, ratio type and
the 38 solvent fractions) on one side. Validate the same way, or your local score will be too optimistic.

In [ ]:
KEY = ["salt", "c", "c units", "solvent ratio type"] + [c for c in NUMERIC if c not in ("T", "c")]
groups = train[KEY].astype(str).agg("|".join, axis=1)
print(groups.nunique(), "formulations in", len(train), "training rows")

### C-2. A validation tool
Which model to use is your decision: a decision tree, a random forest, an MLP (scikit-learn's `MLPRegressor`),
XGBoost or LightGBM through their scikit-learn classes `XGBRegressor` and `LGBMRegressor`, or any other
scikit-learn model. Only scikit-learn models can be submitted. `grouped_rmsle(make_model, X, y, groups)` scores
a model the way the leaderboard does: 5 folds, every formulation on one side, RMSLE (the RMSE of ln(1 + k),
negative predictions counted as 0) averaged over the folds. On this scale the few large k values no longer
dominate the score, so low-conductivity electrolytes count too. `make_model` is a function that returns a new,
untrained model with `fit(X, y)` and `predict(X)`, for example `lambda: make_pipeline(...)`: every fold builds its own
model, so nothing learned on one fold leaks into the score of another. Put the text-column encoding inside the
model (for example a scikit-learn pipeline) so that every fold fits it on its own training rows.

In [ ]:
def grouped_rmsle(make_model, X, y, groups, n_splits=5):
    scores = []
    for fit_idx, val_idx in GroupKFold(n_splits=n_splits).split(X, y, groups=groups):
        model = make_model()   ## a new, untrained model for every fold
        model.fit(X.iloc[fit_idx], y.iloc[fit_idx])
        pred = np.clip(model.predict(X.iloc[val_idx]), 0, None)
        scores.append(np.sqrt(mean_squared_error(np.log1p(y.iloc[val_idx]), np.log1p(pred))))
    return float(np.mean(scores)), float(np.std(scores))

### C-3. Practice 1
Choose two or more candidate models, build each one with its own text-column encoding, and compare them with
`grouped_rmsle`. Then tune the best one with the same function, never with the leaderboard. Your best one
becomes the `make_model` that D-1 trains and saves.

In [ ]:
## Write your code here.
## Order: write make_model() -> mean, std = grouped_rmsle(make_model, X, y, groups) -> compare -> tune the best one

## D. Make a submission

### D-1. Train the final model and save it
Training happens here, in Colab: `fit` adjusts the model's parameters to the training rows, with the settings
you chose in C-3. The server does not train. It loads the saved model and calls `predict` on the hidden test
rows. The cell below builds a new model with `make_model()` (point `make_model` at your best model from C-3),
trains it on every training row and saves it as `model.joblib`. Keep the text-column encoding inside the model
(a pipeline): the server passes the raw test table. The file may be at most 200 MB; `compress=3` keeps it small.

In [ ]:
## Point make_model at your best model from C-3 first (for example make_model = make_forest), then run this cell.
try:
    final_model = make_model()   ## a new, untrained model with your chosen settings
except NameError:
    print("make_model is not defined yet: write it in C-3, then run this cell again.")
else:
    t0 = time.time()
    final_model.fit(X, y)        ## training: the parameters are fitted to every training row
    joblib.dump(final_model, "model.joblib", compress=3)
    print(f"trained in {time.time() - t0:.0f} s; model.joblib is {Path('model.joblib').stat().st_size / 1e6:.1f} MB")

### D-2. Test the file the way the server uses it
This cell loads `model.joblib` from the file, as the server does, and calls `predict` on the training
features. It checks what the server checks (one finite number per row, the file size) and times the load and
the prediction: the server allows 5 minutes. If this cell fails, the server run fails too. The score it prints
is on rows the model was trained on, so it is far better than the leaderboard will be: use it only to see
that the file works, and judge the model with `grouped_rmsle` in C-3. Before D-1 has run, the cell says so.

In [ ]:
path = Path("model.joblib")
if not path.exists():
    print("model.joblib does not exist yet: run D-1 first.")
else:
    t0 = time.time()
    loaded = joblib.load(path)   ## a fresh load from the file: the server sees only this file
    pred = np.asarray(loaded.predict(X), dtype=float).reshape(-1)
    seconds = time.time() - t0
    assert len(pred) == len(X) and np.isfinite(pred).all(), "the server would reject these predictions"
    assert path.stat().st_size <= 200 * 1024 ** 2, "larger than 200 MB: save with compress=3 or use a smaller model"
    rmsle = np.sqrt(mean_squared_error(np.log1p(y), np.log1p(np.clip(pred, 0, None))))
    print(f"loaded and predicted in {seconds:.1f} s; RMSLE on the training rows {rmsle:.3f} (optimistic: the model has seen them)")

### D-3. Submit
1. Run the cell below: in Colab it downloads `model.joblib` to your computer (the site's file picker reads your
   computer, not the Colab runtime). You can also download it from Colab's Files panel.
2. Open the site linked at the top of this notebook, enter your student ID and the 6-digit code from KLMS, and
   upload `model.joblib`. The receipt shows your place in the queue.
3. Press **My submissions** a few minutes later: it shows the status, your public RMSLE or the error message
   of a failed run, and how many of your 5 submissions you have left today. Failed runs do not
   count against that limit.

In [ ]:
try:   ## in Colab: download the file to your computer, then upload it on the site
    from google.colab import files
    files.download("model.joblib")
except ImportError:
    print("Not in Colab: model.joblib is in the working folder.")